# Movie rating classification

## main purpose
Our main purpose  here given a set of movies and set of users rating the movies we want find model that
predict given a movie and user what the user will rate this movie

## data set
first we will install the data set

In [3]:
%pip install kagglehub
import kagglehub

# Download latest version
path = kagglehub.dataset_download("grouplens/movielens-20m-dataset")

print("Path to dataset files:", path)

Note: you may need to restart the kernel to use updated packages.
Path to dataset files: /root/.cache/kagglehub/datasets/grouplens/movielens-20m-dataset/versions/1


In [4]:
from matplotlib import pylab
#from google.colab import drive

import matplotlib.pyplot as plt
import pandas as pd
import sys
!{sys.executable} -m pip install scikit-learn
!{sys.executable} -m pip install seaborn
from sklearn.model_selection import train_test_split
import numpy as np
import seaborn as sns

### this are the files we get from kaggle:

In [50]:
import os
print(os.listdir(path))

['rating.csv', 'genome_tags.csv', 'link.csv', 'tag.csv', 'genome_scores.csv', 'movie.csv']


### rating

In [62]:
ratings = pd.read_csv(os.path.join(path, 'rating.csv'), nrows=1_000_000
)
ratings.head(5)

,userId,movieId,rating,timestamp
0,1,2,3.5,2005-04-02 23:53:47
1,1,29,3.5,2005-04-02 23:31:16
2,1,32,3.5,2005-04-02 23:33:39
3,1,47,3.5,2005-04-02 23:32:07
4,1,50,3.5,2005-04-02 23:29:40


### movies 

In [63]:
movies = pd.read_csv(os.path.join(path, 'movie.csv'))
movies.head(5)

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


## Normalizations

### ratings

In [64]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
ratings["timestamp"] = pd.to_datetime(ratings["timestamp"])
ratings["timestamp"] = (
    ratings["timestamp"].astype(int) // 10**9
)

ratings.head(5)


,userId,movieId,rating,timestamp
0,1,2,3.5,1112486027
1,1,29,3.5,1112484676
2,1,32,3.5,1112484819
3,1,47,3.5,1112484727
4,1,50,3.5,1112484580


### Now we will encode the genere to cuple of features:

In [67]:
from sklearn.preprocessing import MultiLabelBinarizer
mlb = MultiLabelBinarizer()
if "genres" in movies.columns:
    genre_features = mlb.fit_transform(movies['genres'].str.split('|'))
    genre_df = pd.DataFrame(
        genre_features,
        columns=mlb.classes_,
        index=movies.index
    )

    movies = pd.concat(
        [movies[["movieId"]], genre_df],
        axis=1
    )
movies.head(5)

,movieId,(no genres listed),Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,...,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,0,0,1,1,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2,0,0,1,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,3,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,1,0,0,0,0
3,4,0,0,0,0,0,1,0,0,1,...,0,0,0,0,0,1,0,0,0,0
4,5,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
